In [1]:
import mlflow

mlflow.set_experiment("Credit Card Fraud Detection")

print("MLflow experiment configured.")


import joblib

model = joblib.load("creditCardFraud/models/final_xgb_model.joblib")

print("Model type:", type(model).__name__)
print("Is fitted:", hasattr(model, "booster_"))
print("Number of estimators:", model.n_estimators)



MLflow experiment configured.
Model type: XGBClassifier
Is fitted: False
Number of estimators: 200


In [ ]:
mlflow.set_experiment("Credit Card Fraud Detection")

print("Experiment ready.")

Experiment ready.


In [3]:
with mlflow.start_run():
    print("MLflow run started.")

MLflow run started.


In [4]:
with mlflow.start_run():
    mlflow.log_params({
        "model":"XGBoost",
        "n_estimators":200,
        "max_depth":6,
        "learning_rate":0.1,
        "subsample":0.8,
        "colsample_bytree":0.8,
        "scale_pos_weight":599.4761904761905,
    })

    print("Paramaters logged")

Paramaters logged


In [5]:
with mlflow.start_run():
    mlflow.log_metrics({
        "precision":0.9048,
        "recall":0.8000,
        "f1_score":0.8492,
        "roc_auc":0.9761,
        "pr_auc":0.8248,
    })

    print("Metrics logged. ")

Metrics logged. 


In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)



In [7]:
X_train = pd.read_csv("creditCardFraud/data/processed/X_train_scaled.csv")
y_train = pd.read_csv("creditCardFraud/data/processed/y_train.csv").squeeze("columns")

X_test = pd.read_csv("creditCardFraud/data/processed/X_test_scaled.csv")
y_test = pd.read_csv("creditCardFraud/data/processed/y_test.csv").squeeze("columns")

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(226980, 30) (226980,)
(56746, 30) (56746,)


In [8]:
scale_pos_weight = (y_train == 0).sum()/(y_train == 1).sum()

model = XGBClassifier(
    n_estimators = 200,
    max_depth = 6,
    learning_rate = 0.1,
    subsample = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos_weight,
    objective = "binary:logistic",
    eval_metric = "logloss",
    random_state = 42,
    n_jobs = -1,
)

print(f"scale_pos_weight: {scale_pos_weight}")

scale_pos_weight: 599.4761904761905


In [11]:

from sklearn.metrics import average_precision_score, roc_auc_score

current_proba = model.predict_proba(X_test)[:, 1]

print("current PR-AUC:", average_precision_score(y_test, current_proba))
print("Current ROC-AUC:", roc_auc_score(y_test, current_proba))
print("Current probability range:", current_proba.min(), current_proba.max())

current PR-AUC: 0.8248384759168829
Current ROC-AUC: 0.9761103301934559
Current probability range: 4.4280807e-08 0.9999987


In [12]:
with mlflow.start_run(run_name = "Final XGBoost - verified") as run:

    y_proba = model.predict_proba(X_test)[:, 1]

    threshold = 0.2325
    y_pred = (y_proba >= threshold).astype(int)

    metrics = {
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    mlflow.log_params({
        "model": "XGBoost",
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight":scale_pos_weight,
        "threshold":threshold,
    })

    mlflow.log_metrics(metrics)

    mlflow.xgboost.log_model(
        model,
        name = "xgboost_model"
    )


    print("Run ID:", run.info.run_id)
    print(metrics)


Run ID: ee0706d8c74348529b95be2254b47332
{'precision': 0.9047619047619048, 'recall': 0.8, 'f1_score': 0.8491620111731844, 'roc_auc': 0.9761103301934559, 'pr_auc': 0.8248384759168829}


In [13]:
runs = mlflow.search_runs()

runs[[
"run_id",
"metrics.pr_auc",
"metrics.recall",
"metrics.precision",
"metrics.f1_score",
"params.model",
"params.threshold",
]]


,run_id,metrics.pr_auc,metrics.recall,metrics.precision,metrics.f1_score,params.model,params.threshold
0,ee0706d8c74348529b95be2254b47332,0.824838,0.8,0.904762,0.849162,XGBoost,0.2325
1,9abcf7e40d5144c1b0ceb55c427b279b,0.824800,0.8,0.904800,0.849200,None,None
2,8c8b6a25de7d4ff58901672ba057b261,NaN,NaN,NaN,NaN,XGBoost,None
3,b06506f19d5745e5831a6c20ba659f82,NaN,NaN,NaN,NaN,None,None
4,06554cff9a674db98af8c81167fa4752,0.824838,0.8,0.904762,0.849162,XGBoost,0.2325
5,d6ccbf1404f143f6b573f9af1f8e0d9e,0.824800,0.8,0.904800,0.849200,None,None
6,7dc0da2ab3484d5c9d97205895f2787b,NaN,NaN,NaN,NaN,XGBoost,None
7,1da778d349994704b165a20f0a77c99e,NaN,NaN,NaN,NaN,None,None
8,1b1833d56ec4486c87b090ba71f252b7,0.824838,0.8,0.904762,0.849162,XGBoost,0.2325
9,c4d09c679cec44c3bb78a5f573472cf6,0.824800,0.8,0.904800,0.849200,None,None


In [14]:
import os
print("Notebook directory:", os.getcwd())
print("MLflow tracking URI:", mlflow.get_tracking_uri())

Notebook directory: C:\Users\hnkru
MLflow tracking URI: sqlite:///C:/Users/hnkru/mlflow.db


In [15]:
import joblib

MODEL_PATH = "creditCardFraud/models/final_xgb_model.joblib"

model = joblib.load(MODEL_PATH)

# If the saved artifact is unfitted, retrain it
if not hasattr(model, "booster_"):
    model.fit(X_train, y_train)

from sklearn.metrics import average_precision_score, roc_auc_score

current_proba = model.predict_proba(X_test)[:, 1]

print("current PR-AUC:", average_precision_score(y_test, current_proba))
print("Current ROC-AUC:", roc_auc_score(y_test, current_proba))
print("Current probability range:", current_proba.min(), current_proba.max())

current PR-AUC: 0.8248384759168829
Current ROC-AUC: 0.9761103301934559
Current probability range: 4.4280807e-08 0.9999987


In [16]:
import mlflow

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())

MLflow version: 3.16.0
Tracking URI: sqlite:///C:/Users/hnkru/mlflow.db
